# EMRI Waveform Explorer

Set the intrinsic (and optionally extrinsic) source parameters, then run the cells
to generate and compare **fewtrax** and **FEW** waveforms in both the time domain
and the frequency domain.

### Parameters
| Symbol | Meaning | Typical range |
|--------|---------|---------------|
| `M` | Primary BH mass [M☉] | 10⁵ – 10⁷ |
| `mu` | Secondary mass [M☉] | 1 – 100 |
| `a` | Dimensionless spin | 0 – 0.99 |
| `p0` | Initial semi-latus rectum [M] | p_sep + 1 – 20 |
| `e0` | Initial eccentricity | 0 – 0.8 |
| `x0` | Inclination cosine | +1 (prograde) / −1 (retrograde) |
| `T` | Observation time [yr] | 0.01 – 2 |
| `dt` | Sampling interval [s] | 5 – 100 |

In [ ]:
import sys, os
from pathlib import Path

# Make sure the comparison utils are importable
comparison_dir = Path(".").resolve()
if str(comparison_dir) not in sys.path:
    sys.path.insert(0, str(comparison_dir))

import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

from utils import find_data_dir

DATA_DIR = find_data_dir()
print(f"FEW data directory: {DATA_DIR}")

## 1. Set parameters

In [ ]:
# ── Intrinsic parameters ───────────────────────────────────────────────────
M   = 1e6      # primary BH mass  [M☉]
mu  = 10.0     # secondary mass   [M☉]
a   = 0.3      # dimensionless spin
p0  = 10.0     # initial semi-latus rectum  [M]
e0  = 0.4      # initial eccentricity
x0  = 1.0      # inclination cosine (+1 prograde)

# ── Initial orbital phases [rad] ──────────────────────────────────────────
Phi_phi0   = 0.0
Phi_theta0 = 0.0
Phi_r0     = 0.0

# ── Extrinsic parameters ──────────────────────────────────────────────────
dist  = 1.0    # luminosity distance  [Gpc]
qS    = 0.2    # sky colatitude of source  [rad]
phiS  = 0.2    # sky longitude of source   [rad]
qK    = 0.8    # spin-axis polar angle     [rad]
phiK  = 0.8    # spin-axis azimuthal angle [rad]

# ── Duration and sampling ─────────────────────────────────────────────────
T   = 0.1      # observation time  [yr]
dt  = 10.0     # sampling interval [s]

# ── fewtrax settings ──────────────────────────────────────────────────────
mode_threshold = 1e-5   # relative power threshold for mode selection
dense_steps    = 100    # sparse trajectory points fed to the ODE

## 2. Initialise waveform generators

In [ ]:
# ── fewtrax ───────────────────────────────────────────────────────────────
from fewtrax import KerrEccentricEquatorialWaveform

wf_ft = KerrEccentricEquatorialWaveform(
    data_dir=DATA_DIR,
    mode_selection_threshold=mode_threshold,
    dense_steps=dense_steps,
)
print("fewtrax generator ready.")

# ── FEW ───────────────────────────────────────────────────────────────────
from few.waveform import GenerateEMRIWaveform

wf_few = GenerateEMRIWaveform("FastKerrEccentricEquatorialFlux")
print("FEW generator ready.")

## 3. Generate waveforms

In [ ]:
call_kwargs = dict(
    M=M, mu=mu, a=a, p0=p0, e0=e0, x0=x0,
    dist=dist, qS=qS, phiS=phiS, qK=qK, phiK=phiK,
    Phi_phi0=Phi_phi0, Phi_theta0=Phi_theta0, Phi_r0=Phi_r0,
    T=T, dt=dt,
)

# fewtrax
hp_ft, hx_ft, sparse = wf_ft(**call_kwargs, return_sparse=True)
hp_ft = np.asarray(hp_ft)
hx_ft = np.asarray(hx_ft)
n_modes_ft = sparse["teuk_modes"].shape[1]
print(f"fewtrax: {len(hp_ft)} samples,  {n_modes_ft} modes selected")

# FEW
h_few = wf_few(**call_kwargs)
hp_few = np.real(np.asarray(h_few))
hx_few = -np.imag(np.asarray(h_few))
print(f"FEW:     {len(hp_few)} samples")

## 4. Overlap and mismatch

> **Note:** mismatch = 1 − overlap, so a mismatch ≈ 1 (displayed as `1.00e+00`)
> is fully consistent with an overlap ≈ 0.  Large mismatch indicates the
> waveforms are out of phase — the most common cause is a secular drift in
> the accumulated orbital phase between the two integrators.

In [ ]:
def white_noise_overlap(h1, h2):
    """Normalised inner product (flat power spectrum)."""
    n = min(len(h1), len(h2))
    a = h1[:n].astype(float)
    b = h2[:n].astype(float)
    na = np.sqrt(np.dot(a, a))
    nb = np.sqrt(np.dot(b, b))
    if na < 1e-40 or nb < 1e-40:
        return 0.0
    return float(abs(np.dot(a, b)) / (na * nb))

ov_hp = white_noise_overlap(hp_few, hp_ft)
ov_hx = white_noise_overlap(hx_few, hx_ft)
mm_hp = 1.0 - ov_hp
mm_hx = 1.0 - ov_hx

n = min(len(hp_few), len(hp_ft))
rms_hp = float(np.sqrt(np.mean((hp_ft[:n] - hp_few[:n])**2)) /
               (np.sqrt(np.mean(hp_few[:n]**2)) + 1e-40))

print(f"Overlap  h+  = {ov_hp:.6f}    Mismatch = {mm_hp:.4e}")
print(f"Overlap  hx  = {ov_hx:.6f}    Mismatch = {mm_hx:.4e}")
print(f"h+ RMS relative error = {rms_hp:.3e}")

## 5. Time-domain plots

In [ ]:
n_plot = min(len(hp_few), len(hp_ft))
t_days = np.arange(n_plot) * dt / 86400.0

scale = 1.0 / max(float(np.max(np.abs(hp_few[:n_plot]))), 1e-40)

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# h+
axes[0].plot(t_days, hp_few[:n_plot] * scale, label="FEW",     lw=1.2, alpha=0.9)
axes[0].plot(t_days, hp_ft[:n_plot]  * scale, label="fewtrax", lw=1.0, ls="--", alpha=0.8)
axes[0].set_ylabel(r"$h_+ / h_{+,\max}^{\rm FEW}$")
axes[0].set_title(
    fr"$M={M:.0e}\,M_\odot,\; \mu={mu}\,M_\odot,\; a={a},\; p_0={p0},\; e_0={e0}$\n"
    fr"overlap $h_+$ = {ov_hp:.5f},  mismatch = {mm_hp:.2e}"
)
axes[0].legend()

# hx
axes[1].plot(t_days, hx_few[:n_plot] * scale, label="FEW",     lw=1.2, alpha=0.9)
axes[1].plot(t_days, hx_ft[:n_plot]  * scale, label="fewtrax", lw=1.0, ls="--", alpha=0.8)
axes[1].set_ylabel(r"$h_\times / h_{+,\max}^{\rm FEW}$")
axes[1].legend()

# Residual
residual = (hp_ft[:n_plot] - hp_few[:n_plot]) * scale
axes[2].plot(t_days, residual, lw=0.8, color="C2")
axes[2].axhline(0, color="k", lw=0.5, ls="--")
axes[2].set_ylabel(r"$\Delta h_+$  (normalised)")
axes[2].set_xlabel("Time [days]")
axes[2].set_title(f"Residual  (RMS = {rms_hp:.2e})")

plt.tight_layout()
plt.show()

## 6. Frequency-domain plots

A Hann window is applied before the FFT to reduce spectral leakage.  The
signal is zero-padded to the next power of two for better frequency resolution.

In [ ]:
from fewtrax.utils.transforms import to_frequency_domain

def fd(arr, window="hann", zero_pad=True):
    f, h_tilde = to_frequency_domain(
        jnp.asarray(arr[:n_plot], dtype=jnp.float64),
        dt=dt, window=window, zero_pad=zero_pad,
    )
    return np.asarray(f), np.abs(np.asarray(h_tilde))

f_hp_few, amp_hp_few = fd(hp_few)
f_hp_ft,  amp_hp_ft  = fd(hp_ft)
f_hx_few, amp_hx_few = fd(hx_few)
f_hx_ft,  amp_hx_ft  = fd(hx_ft)

f_nyq   = 0.5 / dt
f_min_p = max(f_hp_few[1], 1e-5)

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

axes[0].loglog(f_hp_few, amp_hp_few, label="FEW",     lw=1.2, alpha=0.9)
axes[0].loglog(f_hp_ft,  amp_hp_ft,  label="fewtrax", lw=1.0, ls="--", alpha=0.8)
axes[0].set_xlim(f_min_p, f_nyq)
axes[0].set_ylabel(r"$|\tilde{h}_+(f)|$  [strain/Hz]")
axes[0].set_title("Frequency-domain comparison  (Hann window, zero-padded)")
axes[0].legend()
axes[0].grid(True, which="both", ls=":", alpha=0.4)

axes[1].loglog(f_hx_few, amp_hx_few, label="FEW",     lw=1.2, alpha=0.9)
axes[1].loglog(f_hx_ft,  amp_hx_ft,  label="fewtrax", lw=1.0, ls="--", alpha=0.8)
axes[1].set_xlim(f_min_p, f_nyq)
axes[1].set_ylabel(r"$|\tilde{h}_\times(f)|$  [strain/Hz]")
axes[1].legend()
axes[1].grid(True, which="both", ls=":", alpha=0.4)

# Amplitude ratio
amp_ft_interp = np.interp(f_hp_few, f_hp_ft, amp_hp_ft)
with np.errstate(divide="ignore", invalid="ignore"):
    ratio = np.where(amp_hp_few > 1e-50, amp_ft_interp / amp_hp_few, np.nan)
axes[2].semilogx(f_hp_few, ratio, lw=0.9, color="C2")
axes[2].axhline(1.0, color="k", lw=0.8, ls="--")
axes[2].set_xlim(f_min_p, f_nyq)
axes[2].set_ylim(0, 2)
axes[2].set_ylabel(r"$|\tilde{h}_+^{\rm ft}| / |\tilde{h}_+^{\rm FEW}|$")
axes[2].set_xlabel("Frequency [Hz]")
axes[2].set_title("Spectral amplitude ratio")
axes[2].grid(True, which="both", ls=":", alpha=0.4)

plt.tight_layout()
plt.show()

## 7. Trajectory comparison

Both integrators accumulate the orbital phase **without wrapping** into [0, 2π].
Φ_φ grows monotonically — a linear trend in the difference panel indicates a
constant frequency offset between the two implementations.

In [ ]:
from few.trajectory.inspiral import EMRIInspiral as FEWInspiral
from fewtrax.data.loader import load_flux_data
from fewtrax.trajectory import run_inspiral

# FEW trajectory
traj_few = FEWInspiral(func="KerrEccEqFlux")
res_few = traj_few(M, mu, a, p0, e0, x0, T=T, dt=dt)
t_few_s, p_few, e_few = np.asarray(res_few[0]), np.asarray(res_few[1]), np.asarray(res_few[2])
Phi_phi_few = np.asarray(res_few[4])

# fewtrax trajectory
flux_data = load_flux_data(DATA_DIR)
t_ft_s, p_ft, e_ft, Phi_phi_ft, _, _ = (
    np.asarray(r) for r in run_inspiral(
        a=a, p0=p0, e0=e0, T=T, flux_data=flux_data,
        M=M, mu=mu, dt=dt, x0=x0,
        Phi_phi0=Phi_phi0, Phi_theta0=Phi_theta0, Phi_r0=Phi_r0,
        dense_steps=200,
    )
)
Phi_phi_ft_adj = Phi_phi_ft - Phi_phi0   # align starting phase

t_few_d = t_few_s / 86400.0
t_ft_d  = t_ft_s  / 86400.0

# Common grid for phase difference
t_lo = max(t_few_d[0], t_ft_d[0])
t_hi = min(t_few_d[-1], t_ft_d[-1])
t_c  = np.linspace(t_lo, t_hi, 500)
delta_phi = np.interp(t_c, t_ft_d, Phi_phi_ft_adj) - np.interp(t_c, t_few_d, Phi_phi_few)

fig, axes = plt.subplots(4, 1, figsize=(12, 13), sharex=False)

axes[0].plot(t_few_d, p_few, label="FEW"); axes[0].plot(t_ft_d, p_ft, ls="--", label="fewtrax")
axes[0].set_ylabel(r"$p\;[M]$"); axes[0].legend(); axes[0].set_title("Trajectory comparison")

axes[1].plot(t_few_d, e_few, label="FEW"); axes[1].plot(t_ft_d, e_ft, ls="--", label="fewtrax")
axes[1].set_ylabel(r"$e$"); axes[1].legend()

axes[2].plot(t_few_d, Phi_phi_few,    label="FEW (accumulated)")
axes[2].plot(t_ft_d,  Phi_phi_ft_adj, ls="--", label="fewtrax (accumulated)")
axes[2].set_ylabel(r"$\Phi_\phi$  [rad]  (unwrapped)")
axes[2].legend()

axes[3].plot(t_c, delta_phi, color="C2"); axes[3].axhline(0, color="k", lw=0.5, ls="--")
axes[3].set_ylabel(r"$\Delta\Phi_\phi$  [rad]")
axes[3].set_xlabel("Time [days]")
axes[3].set_title("Phase difference (fewtrax − FEW)")

plt.tight_layout()
plt.show()